# 📊 Polymarket Bot — PnL Audit & Trade Validation

**Purpose:** Parse `paper_trades.log` from the live bot, verify every trade was takeable,
identify open vs closed positions, compute realized/unrealized PnL, cash on hand, and total equity.

Flags positions that may be underwater (losing money).

---
## 1. AWS S3 Download Commands

Run these in your terminal **before** executing this notebook:

```bash
# ── Download bot trade logs (all days) ─────────────────────────────────
aws s3 sync s3://polymarket-data-sobhi/polymarket-bot/daily/ ./data/bot/ --exclude "*" --include "bot_trades_*.log.gz"

# ── Download bot status logs (cash/equity snapshots) ───────────────────
aws s3 sync s3://polymarket-data-sobhi/polymarket-bot/daily/ ./data/bot/ --exclude "*" --include "bot_status_*.log.gz"

# ── Decompress all ─────────────────────────────────────────────────────
gunzip -f ./data/bot/**/*.log.gz 2>/dev/null; gunzip -f ./data/bot/*.log.gz 2>/dev/null

# ── Or download a specific day ─────────────────────────────────────────
# aws s3 cp s3://polymarket-data-sobhi/polymarket-bot/daily/date=2026-03-05/bot_trades_2026-03-05.log.gz .
# gunzip bot_trades_2026-03-05.log.gz
```

If the S3 data isn't available yet, just place your `bot_trades_2026-03-05.log` (or any `paper_trades.log`) in the repo root — the loader below handles both locations.

## 2. Imports & Config

In [ ]:
import re
import glob
import os
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# ── Bot config constants (must match BotConfig.hpp) ──────────────────────────
INITIAL_CAPITAL = 1000.0         # --capital passed to bot
FEE_RATE        = 0.25
FEE_EXPONENT    = 2
SLIPPAGE        = 0.0005         # 5 bps
THRESHOLD       = 0.40           # YES_mid <= this → entry
EXIT_THRESHOLD  = 0.96           # NO_mid  >= this → exit
MAX_ENTRY_ASK   = 0.65           # NO_ask guard

# ── Find trade log files ─────────────────────────────────────────────────────
REPO_ROOT = Path("/workspaces/Polymarket")

candidates = (
    sorted(glob.glob(str(REPO_ROOT / "data" / "bot" / "**" / "bot_trades_*.log"), recursive=True))
    + sorted(glob.glob(str(REPO_ROOT / "bot_trades_*.log")))
    + [str(REPO_ROOT / "paper_trades.log")]
)
candidates = [c for c in candidates if os.path.isfile(c)]
print(f"Found {len(candidates)} trade log file(s):")
for c in candidates:
    print(f"  {c}  ({os.path.getsize(c):,} bytes)")

## 3. Parse Trade Log

Two line formats:
- **BUY**: `{ts} | BUY  NO | {shares}sh | fill={price} | fee={fee} | YES_mid={ym} NO_ask={na} | "{market}"`
- **SELL**: `{ts} | SELL NO | {shares}sh | fill={price} | NO_mid~{nm} | fee={fee} | pnl={pnl} | "{market}"`

In [ ]:
# ── Regex patterns for BUY and SELL lines ─────────────────────────────────────
BUY_RE = re.compile(
    r"^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) UTC \| BUY\s+NO \| "
    r"(?P<shares>[\d.]+)sh \| fill=(?P<fill>[\d.]+) \| fee=(?P<fee>[\d.]+) \| "
    r"YES_mid=(?P<yes_mid>[\d.]+) NO_ask=(?P<no_ask>[\d.]+) \| "
    r'"(?P<market>.+)"'
)

SELL_RE = re.compile(
    r"^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) UTC \| SELL NO \| "
    r"(?P<shares>[\d.]+)sh \| fill=(?P<fill>[\d.]+) \| NO_mid~(?P<no_mid>[\d.]+) \| "
    r"fee=(?P<fee>[\d.]+) \| pnl=(?P<pnl>[+-]?[\d.]+) \| "
    r'"(?P<market>.+)"'
)


def parse_log_files(paths: list[str]) -> pd.DataFrame:
    """Parse one or more bot trade log files into a DataFrame."""
    rows = []
    for path in paths:
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                m = BUY_RE.match(line)
                if m:
                    rows.append({
                        "timestamp": pd.Timestamp(m["ts"], tz="UTC"),
                        "side": "BUY",
                        "market": m["market"],
                        "shares": float(m["shares"]),
                        "fill_price": float(m["fill"]),
                        "fee": float(m["fee"]),
                        "yes_mid": float(m["yes_mid"]),
                        "no_ask": float(m["no_ask"]),
                        "no_mid": np.nan,
                        "logged_pnl": np.nan,
                    })
                    continue

                m = SELL_RE.match(line)
                if m:
                    rows.append({
                        "timestamp": pd.Timestamp(m["ts"], tz="UTC"),
                        "side": "SELL",
                        "market": m["market"],
                        "shares": float(m["shares"]),
                        "fill_price": float(m["fill"]),
                        "fee": float(m["fee"]),
                        "yes_mid": np.nan,
                        "no_ask": np.nan,
                        "no_mid": float(m["no_mid"]),
                        "logged_pnl": float(m["pnl"]),
                    })
                    continue

                print(f"  ⚠ unparsed line: {line[:80]}...")

    df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    return df


trades = parse_log_files(candidates)
print(f"\n✓ Parsed {len(trades)} trades  ({trades['side'].value_counts().to_dict()})")
print(f"  Date range: {trades['timestamp'].min()} → {trades['timestamp'].max()}")
trades.head(10)

## 4. Validate Trade Takeability (Entry Feasibility)

For every **BUY**, check the bot's own entry rules:
1. `YES_mid <= THRESHOLD (0.40)` — the convergence signal
2. `NO_ask <= MAX_ENTRY_ASK (0.65)` — the ask guard
3. `fill_price ≈ NO_ask × (1 + SLIPPAGE)` — price consistency
4. **Fee matches formula**: `fee = shares × FEE_RATE × (p × (1-p))² × p`

In [ ]:
def dynamic_fee_rate(p: float) -> float:
    """Polymarket dynamic taker fee base rate."""
    return FEE_RATE * (p * (1.0 - p)) ** FEE_EXPONENT


def expected_buy_fee(shares: float, fill_price: float) -> float:
    """Expected buy fee in USDC:  shares × base × price."""
    return round(shares * dynamic_fee_rate(fill_price) * fill_price, 4)


def expected_sell_fee(shares: float, fill_price: float) -> float:
    """Expected sell fee in USDC:  shares × base  (no price multiplier)."""
    return round(shares * dynamic_fee_rate(fill_price), 4)


buys = trades[trades["side"] == "BUY"].copy()

# ── Entry condition checks ────────────────────────────────────────────────────
buys["chk_yes_mid"]    = buys["yes_mid"] <= THRESHOLD + 1e-9
buys["chk_no_ask"]     = buys["no_ask"]  <= MAX_ENTRY_ASK + 1e-9
buys["expected_fill"]  = (buys["no_ask"] * (1 + SLIPPAGE)).round(4)
buys["chk_fill_price"] = (buys["fill_price"] - buys["expected_fill"]).abs() < 0.002
buys["expected_fee"]   = buys.apply(lambda r: expected_buy_fee(r["shares"], r["fill_price"]), axis=1)
buys["fee_error"]      = (buys["fee"] - buys["expected_fee"]).abs()
buys["chk_fee"]        = buys["fee_error"] < 0.05  # allow small rounding tolerance

buys["all_valid"] = buys["chk_yes_mid"] & buys["chk_no_ask"] & buys["chk_fill_price"] & buys["chk_fee"]

n_buys = len(buys)
n_valid = buys["all_valid"].sum()
print(f"═══ ENTRY VALIDATION ═══")
print(f"  Total BUY trades:       {n_buys}")
print(f"  All checks passed:      {n_valid} ({100*n_valid/max(n_buys,1):.1f}%)")
print(f"  yes_mid <= {THRESHOLD}:     {buys['chk_yes_mid'].sum()}/{n_buys}")
print(f"  no_ask  <= {MAX_ENTRY_ASK}:     {buys['chk_no_ask'].sum()}/{n_buys}")
print(f"  fill ≈ ask×(1+slip):    {buys['chk_fill_price'].sum()}/{n_buys}")
print(f"  fee matches formula:    {buys['chk_fee'].sum()}/{n_buys}")

# Show any flagged entries
bad_buys = buys[~buys["all_valid"]]
if len(bad_buys) > 0:
    print(f"\n⚠ {len(bad_buys)} flagged BUY trades:")
    display(bad_buys[["timestamp", "market", "shares", "fill_price", "fee",
                       "yes_mid", "no_ask", "expected_fill", "expected_fee",
                       "chk_yes_mid", "chk_no_ask", "chk_fill_price", "chk_fee"]])
else:
    print("\n✅ All BUY trades pass entry feasibility checks")

## 5. Validate Sell-Side (Exit Feasibility)

For every **SELL**, check:
1. `NO_mid >= EXIT_THRESHOLD (0.96)` — the convergence exit trigger
2. `fill_price ≈ NO_bid × (1 - SLIPPAGE)` — since `NO_mid ≈ NO_bid` at these levels
3. Fee matches sell formula: `fee = shares × FEE_RATE × (p × (1-p))²`
4. Logged PnL matches: `proceeds - cost - sell_fee - buy_fee`

In [ ]:
sells = trades[trades["side"] == "SELL"].copy()

# Exit trigger check: NO_mid >= 0.96
sells["chk_no_mid"] = sells["no_mid"] >= EXIT_THRESHOLD - 1e-9

# Fee check
sells["expected_fee"] = sells.apply(lambda r: expected_sell_fee(r["shares"], r["fill_price"]), axis=1)
sells["fee_error"] = (sells["fee"] - sells["expected_fee"]).abs()
sells["chk_fee"] = sells["fee_error"] < 0.05

sells["all_valid"] = sells["chk_no_mid"] & sells["chk_fee"]

n_sells = len(sells)
n_valid_s = sells["all_valid"].sum()
print(f"═══ EXIT VALIDATION ═══")
print(f"  Total SELL trades:       {n_sells}")
print(f"  All checks passed:       {n_valid_s} ({100*n_valid_s/max(n_sells,1):.1f}%)")
print(f"  no_mid >= {EXIT_THRESHOLD}:       {sells['chk_no_mid'].sum()}/{n_sells}")
print(f"  fee matches formula:     {sells['chk_fee'].sum()}/{n_sells}")

bad_sells = sells[~sells["all_valid"]]
if len(bad_sells) > 0:
    print(f"\n⚠ {len(bad_sells)} flagged SELL trades:")
    display(bad_sells[["timestamp", "market", "shares", "fill_price", "fee",
                         "no_mid", "expected_fee", "chk_no_mid", "chk_fee"]])
else:
    print("\n✅ All SELL trades pass exit feasibility checks")

## 6. Match Positions — FIFO BUY/SELL Pairing

Group trades by market name. For each market, pair BUYs to SELLs in FIFO order.
Any unmatched BUY shares become **open positions**.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ClosedPosition:
    market: str
    buy_ts: pd.Timestamp
    sell_ts: pd.Timestamp
    shares: float
    entry_price: float
    exit_price: float
    buy_fee: float
    sell_fee: float
    logged_pnl: float  # from the SELL line
    computed_pnl: float = 0.0

@dataclass
class OpenPosition:
    market: str
    buy_ts: pd.Timestamp
    shares: float
    entry_price: float
    buy_fee: float
    cost_basis: float = 0.0  # shares × entry_price + buy_fee


def match_positions(df: pd.DataFrame):
    """FIFO matching of BUY/SELL by market.  Returns closed + open lists."""
    closed: list[ClosedPosition] = []
    open_pos: list[OpenPosition] = []

    for market, grp in df.groupby("market"):
        # Build a queue of buys
        buy_queue: list[dict] = []
        for _, row in grp.iterrows():
            if row["side"] == "BUY":
                buy_queue.append({
                    "ts": row["timestamp"],
                    "shares": row["shares"],
                    "price": row["fill_price"],
                    "fee": row["fee"],
                })
            elif row["side"] == "SELL":
                sell_shares = row["shares"]
                sell_price  = row["fill_price"]
                sell_fee    = row["fee"]
                sell_ts     = row["timestamp"]
                logged_pnl  = row["logged_pnl"]

                # Match against buy queue (FIFO)
                matched_buy_fee = 0.0
                matched_buy_cost = 0.0
                matched_shares = 0.0

                while sell_shares > 0.01 and buy_queue:
                    b = buy_queue[0]
                    take = min(b["shares"], sell_shares)
                    frac = take / (b["shares"] if b["shares"] > 0 else 1.0)
                    matched_buy_fee += frac * b["fee"]
                    matched_buy_cost += take * b["price"]
                    matched_shares += take

                    sell_shares -= take
                    b["shares"] -= take
                    if b["shares"] < 0.01:
                        buy_queue.pop(0)

                # Compute PnL: proceeds - cost - sell_fee - buy_fee
                proceeds = matched_shares * sell_price
                comp_pnl = proceeds - matched_buy_cost - sell_fee - matched_buy_fee

                closed.append(ClosedPosition(
                    market=market,
                    buy_ts=b["ts"] if buy_queue else sell_ts,
                    sell_ts=sell_ts,
                    shares=matched_shares,
                    entry_price=matched_buy_cost / max(matched_shares, 1e-9),
                    exit_price=sell_price,
                    buy_fee=matched_buy_fee,
                    sell_fee=sell_fee,
                    logged_pnl=logged_pnl,
                    computed_pnl=comp_pnl,
                ))

        # Remaining buys are open
        for b in buy_queue:
            if b["shares"] > 0.01:
                cost = b["shares"] * b["price"] + b["fee"]
                open_pos.append(OpenPosition(
                    market=market,
                    buy_ts=b["ts"],
                    shares=b["shares"],
                    entry_price=b["price"],
                    buy_fee=b["fee"],
                    cost_basis=cost,
                ))

    return closed, open_pos


closed_positions, open_positions = match_positions(trades)

print(f"═══ POSITION MATCHING ═══")
print(f"  Closed (round-trip) positions:  {len(closed_positions)}")
print(f"  Open (still held) positions:    {len(open_positions)}")
print(f"  Total shares still open:        {sum(p.shares for p in open_positions):.1f}")

## 7. Per-Trade Realized PnL

Cross-check the bot's `logged_pnl` against our computed PnL.
Shows per-trade breakdown and flags any discrepancies.

In [ ]:
closed_df = pd.DataFrame([{
    "market":       p.market,
    "buy_ts":       p.buy_ts,
    "sell_ts":      p.sell_ts,
    "shares":       p.shares,
    "entry_price":  p.entry_price,
    "exit_price":   p.exit_price,
    "buy_fee":      p.buy_fee,
    "sell_fee":     p.sell_fee,
    "logged_pnl":   p.logged_pnl,
    "computed_pnl": p.computed_pnl,
} for p in closed_positions]).sort_values("sell_ts").reset_index(drop=True)

# Check PnL consistency
closed_df["pnl_diff"] = (closed_df["logged_pnl"] - closed_df["computed_pnl"]).abs()
closed_df["pnl_match"] = closed_df["pnl_diff"] < 0.50  # allow small float tolerance

n_closed = len(closed_df)
n_match = closed_df["pnl_match"].sum()
print(f"═══ REALIZED PnL — CLOSED POSITIONS ═══")
print(f"  Closed trades:           {n_closed}")
print(f"  PnL matches bot log:     {n_match}/{n_closed}")
print(f"  Total realized (logged): ${closed_df['logged_pnl'].sum():.2f}")
print(f"  Total realized (comp'd): ${closed_df['computed_pnl'].sum():.2f}")
print(f"  Total buy fees:          ${closed_df['buy_fee'].sum():.2f}")
print(f"  Total sell fees:         ${closed_df['sell_fee'].sum():.2f}")

# PnL distribution
print(f"\n  Per-trade PnL stats:")
print(f"    Mean:   ${closed_df['logged_pnl'].mean():.2f}")
print(f"    Median: ${closed_df['logged_pnl'].median():.2f}")
print(f"    Min:    ${closed_df['logged_pnl'].min():.2f}")
print(f"    Max:    ${closed_df['logged_pnl'].max():.2f}")
print(f"    Win rate: {(closed_df['logged_pnl'] > 0).mean()*100:.1f}%")

# Show any mismatches
bad_pnl = closed_df[~closed_df["pnl_match"]]
if len(bad_pnl) > 0:
    print(f"\n⚠ {len(bad_pnl)} PnL mismatches (>$0.50 off):")
    display(bad_pnl[["market", "shares", "entry_price", "exit_price",
                       "logged_pnl", "computed_pnl", "pnl_diff"]].head(20))

closed_df[["sell_ts", "market", "shares", "entry_price", "exit_price",
           "buy_fee", "sell_fee", "logged_pnl", "computed_pnl"]].tail(15)

## 8. PnL by Market / Asset Type

Group realized PnL by market to see which assets are driving profits and which are lagging.

In [ ]:
# ── Extract asset type from market name ────────────────────────────────────────
def extract_asset(market_name: str) -> str:
    """Pull the asset ticker from market question.  e.g. 'Bitcoin', 'Ethereum', 'TSLA'."""
    m = re.match(r"^([\w.]+(?:\s\(\w+\))?)", market_name)
    if m:
        return m.group(1).strip()
    return market_name[:20]

closed_df["asset"] = closed_df["market"].apply(extract_asset)

# Group by asset
asset_pnl = (closed_df.groupby("asset")
    .agg(n_trades=("logged_pnl", "count"),
         total_pnl=("logged_pnl", "sum"),
         avg_pnl=("logged_pnl", "mean"),
         total_fees=("buy_fee", "sum"))
    .sort_values("total_pnl", ascending=False))

asset_pnl["total_fees"] += closed_df.groupby("asset")["sell_fee"].sum()
print("═══ PnL BY ASSET ═══")
display(asset_pnl.style.format({"total_pnl": "${:.2f}", "avg_pnl": "${:.2f}",
                                  "total_fees": "${:.2f}"}))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#2ca02c" if v > 0 else "#d62728" for v in asset_pnl["total_pnl"]]
asset_pnl["total_pnl"].plot(kind="barh", color=colors, ax=ax)
ax.set_xlabel("Realized PnL ($)")
ax.set_title("Realized PnL by Asset")
ax.axvline(0, color="black", lw=0.5)
plt.tight_layout()
plt.show()

## 9. Cash in Hand Over Time

Replay all trades chronologically to track cash balance.
- **BUY**: `cash -= shares × fill_price + fee`
- **SELL**: `cash += shares × fill_price - fee`

In [ ]:
# ── Replay trades to build cash curve ──────────────────────────────────────────
cash = INITIAL_CAPITAL
cash_curve = [{"timestamp": trades["timestamp"].min(), "cash": cash, "event": "start"}]

for _, row in trades.iterrows():
    if row["side"] == "BUY":
        cost = row["shares"] * row["fill_price"] + row["fee"]
        cash -= cost
    else:  # SELL
        proceeds = row["shares"] * row["fill_price"] - row["fee"]
        cash += proceeds

    cash_curve.append({
        "timestamp": row["timestamp"],
        "cash": cash,
        "event": row["side"],
    })

cash_df = pd.DataFrame(cash_curve)

print(f"═══ CASH BALANCE REPLAY ═══")
print(f"  Starting capital:    ${INITIAL_CAPITAL:.2f}")
print(f"  Final cash on hand:  ${cash:.2f}")
print(f"  Lowest cash point:   ${cash_df['cash'].min():.2f}")
print(f"  Highest cash point:  ${cash_df['cash'].max():.2f}")

# Plot cash over time
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(cash_df["timestamp"], cash_df["cash"], lw=1.0, color="#1f77b4")
ax.fill_between(cash_df["timestamp"], cash_df["cash"], alpha=0.15, color="#1f77b4")
ax.set_ylabel("Cash ($)")
ax.set_title(f"Cash Balance Over Time  (started ${INITIAL_CAPITAL:.0f})")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
ax.axhline(INITIAL_CAPITAL, color="gray", ls="--", lw=0.7, label="initial")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Open Positions — Mark-to-Market & Risk Assessment

These are BUY trades with no matching SELL yet. The bot is still holding these NO shares.

**Scenarios for open positions:**
- **Best case**: Market resolves at NO=1.0 → payout = shares × $1.00
- **Worst case**: Market resolves at NO=0.0 → total loss of cost basis
- **Current MTM**: Use entry_price as fallback (no live book data in log)

The key question: **how many open $$ are at risk?**

In [ ]:
open_df = pd.DataFrame([{
    "market":       p.market,
    "buy_ts":       p.buy_ts,
    "shares":       p.shares,
    "entry_price":  p.entry_price,
    "buy_fee":      p.buy_fee,
    "cost_basis":   p.cost_basis,
} for p in open_positions]).sort_values("buy_ts").reset_index(drop=True)

if len(open_df) > 0:
    open_df["asset"] = open_df["market"].apply(extract_asset)

    # Since we don't have live book data, estimate scenarios
    # Best case: NO resolves at 1.0 → payout = shares × 1.0
    open_df["payout_best"] = open_df["shares"] * 1.0
    open_df["pnl_best"]    = open_df["payout_best"] - open_df["cost_basis"]

    # Breakeven: what NO price do we need to break even?
    # breakeven = cost_basis / shares
    open_df["breakeven_no"] = open_df["cost_basis"] / open_df["shares"]

    # Worst case: NO resolves at 0.0 → loss = cost_basis
    open_df["pnl_worst"]   = -open_df["cost_basis"]

    # Estimate unrealized PnL at entry_price (conservative: MTM = entry)
    # Exit sell fee would be: shares × dynamic_fee_rate(exit_price)
    # At ~0.95: dynamic_fee_rate(0.95) = 0.25 × (0.95 × 0.05)^2 = 0.25 × 0.002256 = 0.000564
    est_exit = 0.95  # conservative estimate if we could sell now
    open_df["est_exit_fee"] = open_df["shares"] * dynamic_fee_rate(est_exit)
    open_df["est_pnl"]     = (open_df["shares"] * est_exit
                               - open_df["cost_basis"]
                               - open_df["est_exit_fee"])

    # Time since entry (hours)
    now = pd.Timestamp.now(tz="UTC")
    open_df["hours_held"] = (now - open_df["buy_ts"]).dt.total_seconds() / 3600

    total_cost = open_df["cost_basis"].sum()
    total_best = open_df["pnl_best"].sum()
    total_est  = open_df["est_pnl"].sum()

    print(f"═══ OPEN POSITIONS ═══")
    print(f"  Count:                  {len(open_df)}")
    print(f"  Total cost basis:       ${total_cost:.2f}")
    print(f"  Total shares held:      {open_df['shares'].sum():.1f}")
    print(f"  Best-case PnL (NO=1):   ${total_best:+.2f}")
    print(f"  Est. PnL (NO=0.95):     ${total_est:+.2f}")
    print(f"  Worst-case loss (NO=0): ${open_df['pnl_worst'].sum():.2f}")
    print(f"  Avg hours held:         {open_df['hours_held'].mean():.1f}h")

    display(open_df[["market", "buy_ts", "shares", "entry_price",
                      "cost_basis", "breakeven_no", "est_pnl", "hours_held"]]
            .style.format({"entry_price": "{:.4f}", "cost_basis": "${:.2f}",
                            "breakeven_no": "{:.4f}", "est_pnl": "${:+.2f}",
                            "hours_held": "{:.1f}h"}))
else:
    print("✅ No open positions — all trades are fully closed!")

## 11. Total Equity = Cash + Open Position Value

$\text{Equity} = \text{Cash} + \sum_{i}(\text{shares}_i \times \hat{p}_{\text{exit}})$

Where $\hat{p}_{\text{exit}}$ is the estimated exit price for open NO tokens.

In [ ]:
# ── Compute equity under different scenarios ──────────────────────────────────
final_cash = cash  # from the cash replay above

if len(open_df) > 0:
    open_mtm_entry = (open_df["shares"] * open_df["entry_price"]).sum()
    open_mtm_095   = (open_df["shares"] * 0.95).sum()
    open_mtm_best  = (open_df["shares"] * 1.0).sum()
else:
    open_mtm_entry = 0.0
    open_mtm_095 = 0.0
    open_mtm_best = 0.0

total_realized = closed_df["logged_pnl"].sum() if len(closed_df) > 0 else 0.0
total_fees_all = (
    (closed_df["buy_fee"].sum() + closed_df["sell_fee"].sum() if len(closed_df) > 0 else 0.0)
    + (open_df["buy_fee"].sum() if len(open_df) > 0 else 0.0)
)

equity_at_entry = final_cash + open_mtm_entry
equity_at_095   = final_cash + open_mtm_095
equity_at_best  = final_cash + open_mtm_best

print("╔══════════════════════════════════════════════════════╗")
print("║           TOTAL EQUITY BREAKDOWN                    ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Starting capital:       ${INITIAL_CAPITAL:>10.2f}             ║")
print(f"║                                                      ║")
print(f"║  Total realized PnL:     ${total_realized:>+10.2f}             ║")
print(f"║  Total fees paid:        ${total_fees_all:>10.2f}             ║")
print(f"║                                                      ║")
print(f"║  Cash on hand:           ${final_cash:>10.2f}             ║")
print(f"║  Open positions (#):     {len(open_df):>10d}             ║")
print(f"║  Open cost basis:        ${open_df['cost_basis'].sum() if len(open_df) > 0 else 0:>10.2f}             ║")
print(f"║                                                      ║")
print(f"║  ── Equity Scenarios ──                              ║")
print(f"║  @ entry price (cons.):  ${equity_at_entry:>10.2f}             ║")
print(f"║  @ NO=0.95 (likely):     ${equity_at_095:>10.2f}             ║")
print(f"║  @ NO=1.00 (best):       ${equity_at_best:>10.2f}             ║")
print(f"║                                                      ║")
print(f"║  Equity return:          {(equity_at_095/INITIAL_CAPITAL - 1)*100:>+10.1f}%             ║")
print("╚══════════════════════════════════════════════════════╝")

## 12. Flag Risky Open Positions

Positions that are concerning:
- **Stale**: Held for too long (market may have expired without the bot noticing)
- **Stock markets**: Daily up/down markets resolve at market close — if it's past close, NO might go to 0
- **Large exposure**: Single positions that represent >10% of equity

In [ ]:
if len(open_df) > 0:
    # ── Categorize risk ───────────────────────────────────────────────────────
    open_df["pct_of_equity"] = open_df["cost_basis"] / max(equity_at_095, 1) * 100

    # Is it a stock market? (These resolve at 4pm ET, so "Up or Down on March 5?"
    # means we know the outcome already — if it went up, NO=0 → total loss)
    stock_pattern = re.compile(r"\((\w+)\).*Up or Down on")
    open_df["is_stock"] = open_df["market"].apply(lambda m: bool(stock_pattern.search(m)))

    # Is it a crypto 5-min bucket that already expired?
    # (format: "Bitcoin Up or Down - March 4, 9:10PM-9:15PM ET")
    open_df["is_crypto_bucket"] = open_df["market"].apply(
        lambda m: bool(re.search(r"\d+:\d+[AP]M-\d+:\d+[AP]M ET", m)))

    # Large position flag
    open_df["large_pos"] = open_df["pct_of_equity"] > 10.0

    # Holdings > 6 hours old might be stale (5-min crypto buckets resolve fast)
    open_df["stale"] = open_df["hours_held"] > 6.0

    # Risk score (higher = worse)
    open_df["risk_score"] = (
        open_df["is_stock"].astype(int) * 3
        + open_df["stale"].astype(int) * 2
        + open_df["large_pos"].astype(int) * 1
    )

    risky = open_df[open_df["risk_score"] > 0].sort_values("risk_score", ascending=False)

    print(f"═══ RISK FLAGS ═══")
    print(f"  Total open:             {len(open_df)}")
    print(f"  Stock day-markets:      {open_df['is_stock'].sum()}")
    print(f"  Crypto 5-min buckets:   {open_df['is_crypto_bucket'].sum()}")
    print(f"  Stale (>6h):            {open_df['stale'].sum()}")
    print(f"  Large (>10% equity):    {open_df['large_pos'].sum()}")
    print(f"  Flagged positions:      {len(risky)}")

    if len(risky) > 0:
        print(f"\n⚠ FLAGGED POSITIONS (sorted by risk):")
        display(risky[["market", "shares", "entry_price", "cost_basis",
                         "hours_held", "pct_of_equity", "is_stock", "stale",
                         "risk_score"]]
                .style.format({"entry_price": "{:.4f}", "cost_basis": "${:.2f}",
                                "hours_held": "{:.1f}h", "pct_of_equity": "{:.1f}%"})
                .background_gradient(subset=["risk_score"], cmap="YlOrRd"))

    # Summarize exposure by category
    print(f"\n  ── Exposure by category ──")
    stock_exp = open_df.loc[open_df["is_stock"], "cost_basis"].sum()
    crypto_exp = open_df.loc[~open_df["is_stock"], "cost_basis"].sum()
    print(f"    Stock daily markets: ${stock_exp:.2f}")
    print(f"    Crypto markets:     ${crypto_exp:.2f}")
else:
    print("✅ No open positions to flag")

## 13. Summary Dashboard

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  MULTI-PANEL DASHBOARD
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Panel 1 (top-left): Cumulative PnL curve ─────────────────────────────────
ax = axes[0, 0]
if len(closed_df) > 0:
    cum_pnl = closed_df["logged_pnl"].cumsum()
    ax.plot(closed_df["sell_ts"], cum_pnl, lw=1.5, color="#2ca02c")
    ax.fill_between(closed_df["sell_ts"], cum_pnl, alpha=0.15, color="#2ca02c")
    ax.set_ylabel("Cumulative PnL ($)")
    ax.set_title(f"Realized PnL Curve — Total ${cum_pnl.iloc[-1]:+,.2f}")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
else:
    ax.text(0.5, 0.5, "No closed trades", ha="center", va="center", transform=ax.transAxes)

# ── Panel 2 (top-right): PnL distribution ────────────────────────────────────
ax = axes[0, 1]
if len(closed_df) > 0:
    pnl_vals = closed_df["logged_pnl"]
    ax.hist(pnl_vals, bins=30, color="#1f77b4", edgecolor="white", alpha=0.8)
    ax.axvline(pnl_vals.mean(), color="red", ls="--", lw=1, label=f"mean ${pnl_vals.mean():.2f}")
    ax.axvline(pnl_vals.median(), color="orange", ls="--", lw=1, label=f"median ${pnl_vals.median():.2f}")
    ax.set_xlabel("PnL per trade ($)")
    ax.set_title(f"PnL Distribution — {(pnl_vals > 0).mean()*100:.0f}% win rate")
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, "No closed trades", ha="center", va="center", transform=ax.transAxes)

# ── Panel 3 (bottom-left): Cash over time ────────────────────────────────────
ax = axes[1, 0]
ax.plot(cash_df["timestamp"], cash_df["cash"], lw=1.0, color="#1f77b4")
ax.fill_between(cash_df["timestamp"], cash_df["cash"], alpha=0.15, color="#1f77b4")
ax.axhline(INITIAL_CAPITAL, color="gray", ls="--", lw=0.7)
ax.set_ylabel("Cash ($)")
ax.set_title(f"Cash Balance — Final ${cash:.2f}")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))

# ── Panel 4 (bottom-right): Open position exposure ───────────────────────────
ax = axes[1, 1]
if len(open_df) > 0:
    # Bar chart of top open positions by cost basis
    top_open = open_df.nlargest(15, "cost_basis")
    short_names = [m[:35] + "..." if len(m) > 35 else m for m in top_open["market"]]
    colors = ["#d62728" if s else "#ff7f0e" for s in top_open["is_stock"]]
    ax.barh(range(len(top_open)), top_open["cost_basis"], color=colors)
    ax.set_yticks(range(len(top_open)))
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel("Cost Basis ($)")
    ax.set_title(f"Top Open Positions — {len(open_df)} total (red=stock)")
    ax.invert_yaxis()
else:
    ax.text(0.5, 0.5, "No open positions", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Open Positions — None")

plt.suptitle("Polymarket Bot — PnL Audit Dashboard", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Final summary table ──────────────────────────────────────────────────────
n_total = len(trades)
n_buys_total = (trades["side"] == "BUY").sum()
n_sells_total = (trades["side"] == "SELL").sum()
avg_win = closed_df.loc[closed_df["logged_pnl"] > 0, "logged_pnl"].mean() if len(closed_df) > 0 else 0
avg_loss = closed_df.loc[closed_df["logged_pnl"] <= 0, "logged_pnl"].mean() if len(closed_df) > 0 else 0

summary = pd.DataFrame({
    "Metric": [
        "Total trades logged", "BUY trades", "SELL trades",
        "Closed round-trips", "Open positions",
        "Entry checks passed", "Exit checks passed",
        "Realized PnL", "Total fees",
        "Cash on hand", "Open cost basis",
        "Equity @ NO=0.95", "Equity return",
        "Win rate", "Avg win", "Avg loss",
    ],
    "Value": [
        f"{n_total}", f"{n_buys_total}", f"{n_sells_total}",
        f"{len(closed_df)}", f"{len(open_df)}",
        f"{buys['all_valid'].sum()}/{n_buys}", f"{sells['all_valid'].sum()}/{n_sells}",
        f"${total_realized:+,.2f}", f"${total_fees_all:,.2f}",
        f"${final_cash:,.2f}",
        f"${open_df['cost_basis'].sum() if len(open_df) > 0 else 0:,.2f}",
        f"${equity_at_095:,.2f}",
        f"{(equity_at_095/INITIAL_CAPITAL - 1)*100:+.1f}%",
        f"{(closed_df['logged_pnl'] > 0).mean()*100:.1f}%" if len(closed_df) > 0 else "N/A",
        f"${avg_win:,.2f}", f"${avg_loss:,.2f}",
    ],
})
display(summary.style.hide(axis="index"))